In [1]:
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:qwen/qwen3-32b")

response=model.invoke("CEO of Meta")
response

AIMessage(content="<think>\nOkay, the user is asking about the CEO of Meta. Let me start by recalling the current leadership. As of now, Meta's CEO is Mark Zuckerberg. But wait, there was a change in October 2023, right? Let me confirm that. I think he stepped down from the role of CEO, but maybe he's still the executive chair. Let me check the dates.\n\nWait, no, I think he became the CEO after Steve Jobs, but no, that's Apple. Meta's CEO history: the company was originally Facebook, founded by Mark Zuckerberg. He was the CEO until 2023 when he stepped down as CEO but became the executive chair. Then in October 2023, he stepped down from the chair as well, and the current CEO is now someone else. Wait, maybe the user is asking as of when? The user didn't specify a date, so they might be asking for the current CEO as of today.\n\nLet me verify. As of October 2023, Mark Zuckerberg stepped down as CEO of Meta and became the executive chair. The new CEO is now someone else. Wait, no, I th

## Pydantic

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    director:str=Field(description="The director of the movie")
    Year:int=Field(description="the movie released year")
    rating:float=Field(description="the rating of the movie")

model_structured_op=model.with_structured_output(Movie)
response=model_structured_op.invoke("provide detail about the movie vikram")
response

Movie(title='Vikram', director='Lokesh Kanagaraj', Year=2022, rating=8.2)

In [5]:
response=model.invoke("provide detail about the movie vikram")
response

AIMessage(content='<think>\nOkay, so I need to provide details about the movie "Vikram". Let me start by recalling what I know. "Vikram" is a recent Indian film, right? It\'s a Tamil movie starring Kamal Haasan in the titular role. The director is Lokesh Kanagaraj, who\'s known for other works like "Kavan" and "Kanchana 3". The movie has been quite a big deal, I think. It\'s a political thriller with some action elements. Let me check if that\'s correct.\n\nFirst, the release date: I believe it was 2022, but maybe 2021? Wait, I\'m a bit confused. Maybe it was delayed. Let me think. Kamal Haasan is a veteran actor, so his movies often have a lot of buildup. The movie\'s title "Vikram" is also the name of the character he plays. The plot involves a police officer who takes on organized crime. There\'s a rivalry between the police and a criminal syndicate led by someone like Madathiparambil Gnanasekar, played by someone like Vijay Sethupathi? Or maybe someone else? Wait, Vijay Sethupathi 

### Raw AI Message + Structured o/p

In [8]:
class Movie(BaseModel):
    """The movie details..."""
    title:str=Field(description="the title of the movie")
    director:str=Field(description="The director of the movie")
    Year:int=Field(description="the movie released year")
    rating:float=Field(description="the rating of the movie")

model_structured_op=model.with_structured_output(Movie,include_raw=True)
response=model_structured_op.invoke("provide detail about the movie vikram")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Vikram". Let me check what tools I have available. There\'s a Movie function that requires title, director, Year, and rating. I need to make sure I have all that information. I know the title is "Vikram", but I need to recall the director, release year, and rating. Let me think... Vikram is a Tamil movie directed by Lokesh Kanagaraj. It was released in 2022. The rating might be around 8.5 or so. Let me verify that. Yes, on IMDb it\'s rated 8.5. So I should structure the tool call with those details. Make sure the parameters are in the correct format: title as a string, director as a string, Year as an integer, and rating as a number. Alright, that\'s all the required fields. Let me put it into the JSON object as specified.\n', 'tool_calls': [{'id': 'yrfcfr7zj', 'function': {'arguments': '{"Year":2022,"director":"Lokesh Kanagaraj","rating":8.5,"title":"Vikram"}', 

### nested structure

In [9]:
class characters(BaseModel):
    name:str
    role:str

class bookdetails(BaseModel):
    "The details of the book"
    title:str
    author:str
    year:int
    Members: list[characters]
    genres: list[str]

model_structured_op=model.with_structured_output(bookdetails)
response=model_structured_op.invoke("provide details about 2 states book")
response

bookdetails(title='2 States', author='Chetan Bhagat', year=2009, Members=[characters(name='Ishan', role='Protagonist'), characters(name='Chaitali', role='Protagonist')], genres=['Romance', 'Coming-of-age'])

### TypedDict

In [20]:
from typing_extensions import Annotated,TypedDict

class Moviedet(TypedDict):
    "Details of the Movie"
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"Movie released year"]
    director:Annotated[str,...,"Director of the movie"]
    rating:Annotated[float,...,"Rating of the movie"]
    Actor:Annotated[str,...,"Actor of the movie"]

model_withtypeddict=model.with_structured_output(Moviedet)
response=model_withtypeddict.invoke("Provide detail about Ghilli Movie")
response

{'Actor': 'Vijay',
 'director': 'S. P. Muthuraj',
 'rating': 7.5,
 'title': 'Ghilli',
 'year': 2004}

In [24]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetail(TypedDict):
    title:str
    cast:list[Actor]
    geners:list[str]
    year:int
    budget:float | None =Field(None,description="Budget of the movie")

model_with_typed_dict=model.with_structured_output(MovieDetail)
response=model_with_typed_dict.invoke("provide detail of movie jawan")
response

{'budget': 200000000,
 'cast': [{'name': 'Shah Rukh Khan', 'role': 'Jawan (Protagonist)'},
  {'name': 'Nayanthara', 'role': 'Female Lead'},
  {'name': 'Vijay Sethupathi', 'role': 'Antagonist'}],
 'geners': ['Action', 'Drama', 'Romance'],
 'title': 'Jawan',
 'year': 2023}

In [25]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClass

In [31]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [33]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    "Contact detail of the person"
    name:str=Field(description="Name of the person")
    phone:str=Field(description="Phone number of the person")
    email:str=Field(description="Email id of the person")

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result=agent.invoke({"messages":[{"role":"user",
               "content":"Extract info from this: Kanimozhi 964457834678 kanimozhipjk483@gmail.com "}]
})
result

{'messages': [HumanMessage(content='Extract info from this: Kanimozhi 964457834678 kanimozhipjk483@gmail.com ', additional_kwargs={}, response_metadata={}, id='eab3969d-a548-4f92-be3f-15b325e52733'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user provided a string with some information and wants me to extract it. The input is "Kanimozhi 964457834678 kanimozhipjk483@gmail.com". \n\nFirst, I need to identify which parts correspond to the name, phone, and email. The name is probably "Kanimozhi" since that\'s the first word. Then, the next part is a number: 964457834678. That looks like a phone number. The last part is an email address: kanimozhipjk483@gmail.com.\n\nNow, checking the tool provided. The ContactInfo function requires name, phone, and email. All three are present here. I need to make sure the phone number is a string, which it is. The email format looks valid. \n\nWait, the phone number has 12 digits. Usually, phone numbers are 10 

In [34]:
result['structured_response']

ContactInfo(name='Kanimozhi', phone='964457834678', email='kanimozhipjk483@gmail.com')

In [43]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    "Contact detail of the person"
    name:str
    phone:str
    email:str

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result=agent.invoke({"messages":[{"role":"user",
               "content":"Extract contact info from this: Kanimozhi, 9644578678, kanimozhipjk483@gmail.com "}]
})
result

{'messages': [HumanMessage(content='Extract contact info from this: Kanimozhi, 9644578678, kanimozhipjk483@gmail.com ', additional_kwargs={}, response_metadata={}, id='e5f2a1f0-b978-4f6e-8db9-338702949df9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "Kanimozhi, 9644578678, kanimozhipjk483@gmail.com". \n\nFirst, I need to identify the different parts. The first part is the name, which is "Kanimozhi". Then there\'s the phone number, "9644578678", and the email address "kanimozhipjk483@gmail.com". \n\nI should check the parameters required for the ContactInfo function. The required fields are name, phone, and email. All three are present here. The phone number doesn\'t have any special characters, just numbers, so that\'s straightforward. The email looks valid as well. \n\nI need to make sure the parameters are correctly assigned. The name is "Kanimozhi", phone is "9644578678", a

In [44]:
result['structured_response']

{'name': 'Kanimozhi',
 'phone': '9644578678',
 'email': 'kanimozhipjk483@gmail.com'}

In [45]:
#Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name:str
    email:str
    phone:str

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result=agent.invoke({"messages":[{"role":"user",
                           "content":"Extract contact info : kanimozhi 965678908 kanimozhipjk483@gmail.com"}]
                           })
result


{'messages': [HumanMessage(content='Extract contact info : kanimozhi 965678908 kanimozhipjk483@gmail.com', additional_kwargs={}, response_metadata={}, id='3fff8a6e-f8a8-4674-a46e-629a34fac1fc'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text. The input is "kanimozhi 965678908 kanimozhipjk483@gmail.com". \n\nFirst, I need to identify the name, phone number, and email address. The name appears to be "kanimozhi". The phone number is "965678908", which seems a bit short. Usually, phone numbers are longer, maybe 10 digits? But maybe it\'s correct as given. The email is "kanimozhipjk483@gmail.com".\n\nI should check if all required fields are present. The function ContactInfo requires name, email, and phone. All three are here. So I\'ll map them into the function parameters. The name is "kanimozhi", email is the provided email, and phone is the number given. Even though the phone number i

In [46]:
result['structured_response']

ContactInfo(name='kanimozhi', email='kanimozhipjk483@gmail.com', phone='965678908')